# Phase 2S - Segmentation with real ground truth
### Retrain the U-Net on MSD Task06_Lung, report Dice / IoU on held-out patients

**Run on:** Colab, free T4 GPU. Budget ~70-100 min (download ~15 min, preprocessing
~10 min, two training runs ~10 min each, evaluation ~15 min).

---

### Why this notebook exists

IQ-OTH/NCCD ships class labels only, with no per-pixel annotations, so Dice and IoU
cannot be computed there at all (finding F4). This notebook obtains those numbers from
a dataset that has voxel-level expert masks, and produces a segmenter whose training is
fully documented.

### Why MSD Task06 and not LUNA16

LUNA16's nodule annotations are **centroid + diameter** (1186 nodules >=3 mm at
3-of-4 radiologist consensus). Scoring Dice there would mean drawing spherical
pseudo-masks from those coordinates and measuring the segmenter against our own
drawing. LUNA16's real masks segment **lung fields**, not lesions. MSD Task06_Lung has
genuine per-voxel expert tumour masks: 96 thin-section CT volumes of NSCLC patients,
of which the **64 training cases carry released labels**.

LUNA16 is used in the companion notebook for **detection** sensitivity and FROC, the
metric its annotation format genuinely supports.

### Why retrain rather than evaluate the existing checkpoint

There is no training code and no training masks for `UNet_best_Model_checkpoint.h5`.
Its provenance is unknown and cannot be described honestly in a Method section.
Retraining fixes that. The old checkpoint is still scored on the same held-out volumes
as the first row of the segmentation ablation.

### Two caveats that belong in the paper

1. MSD Task06 labels are **tumour** masks (NSCLC); the pipeline targets **nodules**.
   Related, not identical.
2. Dice is measured on MSD while classification is measured on IQ-OTH/NCCD, across
   different domains (3D thin-section CT in Hounsfield units against 8-bit JPEG-like
   slices). The Dice figure does **not** transfer as a guarantee of segmentation quality
   on IQ-OTH/NCCD. Section 10 shows the transfer qualitatively so the gap is visible
   rather than assumed.

## 1. Environment

In [ ]:
import subprocess, sys
for pkg in ["nibabel", "opencv-python-headless", "tabulate"]:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False)

import tensorflow as tf, keras
gpus = tf.config.list_physical_devices("GPU")
print("tensorflow:", tf.__version__, "| keras:", keras.__version__)
print("GPUs      :", gpus)
assert gpus, "No GPU. Runtime > Change runtime type > T4 GPU, then rerun."

In [ ]:
import os, json, random, glob, shutil, time
import numpy as np
import pandas as pd
import cv2
import nibabel as nib
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

# ---- configuration -------------------------------------------------------
RAW_SIZE     = 512    # slices are cached once at this size, then downsampled per config
NEG_RATIO    = 0.3    # tumour-free slices per tumour-bearing slice
EPOCHS       = 40
BASE_FILTERS = 32
MIXED_PRECISION = True

LUNG = {"level": -600, "width": 1500, "wdesc": "lung window"}
SOFT = {"level": 40,   "width": 400,  "wdesc": "soft-tissue window"}

# v2 trained both windows and the lung window won on validation (0.325 against 0.262),
# so the earlier hypothesis that the lung window was clipping away tumour contrast was
# wrong; the metric was the whole problem. v3 therefore keeps the lung window and
# spends the compute on resolution instead, which the v2 error pattern points at:
# the model over-segments badly (one volume predicted 57x its true tumour volume),
# and 256px downsampling of a target occupying 0.3% of the image is a likely cause.
CONFIGS = [
    {"tag": "lung256", "size": 256, "batch": 8, **LUNG},
    {"tag": "lung512", "size": 512, "batch": 4, **LUNG},
]
PRIOR_RESULTS = {"soft256_val_dice": 0.2616}   # carried from the v2 run, reported as-is

# Selection swept to 0.97. v2 swept only to 0.70 and validation Dice was still rising
# at that boundary, so its operating point was chosen at the edge of the search space
# and its test figure understates what the same weights can do.
THRS = [0.3, 0.5, 0.7, 0.8, 0.9, 0.95, 0.97]
# --------------------------------------------------------------------------

if MIXED_PRECISION:
    keras.mixed_precision.set_global_policy("mixed_float16")
    print("mixed precision:", keras.mixed_precision.global_policy().name)

RESULTS_DIR = "/content/fyp_phase2s_results"
os.makedirs(RESULTS_DIR, exist_ok=True)
RESULTS = {"seed": SEED, "config": {
    "raw_size": RAW_SIZE, "neg_ratio": NEG_RATIO, "epochs": EPOCHS,
    "base_filters": BASE_FILTERS, "configs": CONFIGS, "thresholds": THRS,
    "prior_results": PRIOR_RESULTS}}

def save_json():
    with open(f"{RESULTS_DIR}/results.json", "w") as f:
        json.dump(RESULTS, f, indent=2, default=float)
print("results ->", RESULTS_DIR)

## 2. Download MSD Task06_Lung

Streamed from the MONAI S3 mirror directly into `tar`, so the 9.2 GB archive is never
written to disk. Only `imagesTr` and `labelsTr` are extracted (~5.7 GB); the unlabelled
`imagesTs` is skipped. AppleDouble entries (`._*`) are filtered out, as they are not
valid NIfTI and crash nibabel.

In [ ]:
MSD_URL = "https://msd-for-monai.s3-us-west-2.amazonaws.com/Task06_Lung.tar"
ROOT = "/content/msd"
os.makedirs(ROOT, exist_ok=True)

if not os.path.isdir(f"{ROOT}/Task06_Lung/labelsTr"):
    cmd = (f"curl -L --retry 3 --fail '{MSD_URL}' | "
           f"tar -x -C {ROOT} --wildcards --exclude='._*' "
           f"'Task06_Lung/imagesTr/*' 'Task06_Lung/labelsTr/*'")
    t0 = time.time()
    subprocess.run(["bash", "-c", cmd], check=True)
    print(f"downloaded + extracted in {(time.time()-t0)/60:.1f} min")
else:
    print("already present")

IMG_DIR = f"{ROOT}/Task06_Lung/imagesTr"
LBL_DIR = f"{ROOT}/Task06_Lung/labelsTr"
print("images:", len(os.listdir(IMG_DIR)), "| labels:", len(os.listdir(LBL_DIR)))

### 2.1 Patient-level split

Split **by volume (patient)**, never by slice. Adjacent axial slices of one patient are
near-duplicates, so slice-level splitting leaks them across train and test and inflates
Dice substantially. Phase 0 measured exactly this effect on IQ-OTH/NCCD (11.8 accuracy
points), and the same mistake is not repeated here.

In [ ]:
cases = sorted(f for f in os.listdir(LBL_DIR) if f.endswith(".nii.gz"))
cases = [c for c in cases if os.path.exists(os.path.join(IMG_DIR, c))]
print("labelled volumes:", len(cases))

rng = np.random.RandomState(SEED)
perm = rng.permutation(len(cases))
n_te = max(1, int(round(0.20 * len(cases))))
n_va = max(1, int(round(0.15 * len(cases))))
test_c  = [cases[i] for i in perm[:n_te]]
val_c   = [cases[i] for i in perm[n_te:n_te + n_va]]
train_c = [cases[i] for i in perm[n_te + n_va:]]

split_of = {**{c: "train" for c in train_c}, **{c: "val" for c in val_c},
            **{c: "test" for c in test_c}}
pd.DataFrame({"case": cases, "split": [split_of[c] for c in cases]}) \
  .to_csv(f"{RESULTS_DIR}/msd_split_seed{SEED}.csv", index=False)

print(f"train {len(train_c)} | val {len(val_c)} | test {len(test_c)}")
RESULTS["split"] = {"train": train_c, "val": val_c, "test": test_c}
save_json()

## 3. Preprocessing

1. **HU windowing** to one of the two windows under test, rescaled to 8-bit.
2. **CLAHE** (clipLimit 2.0, 8x8 tiles), the same step the IQ-OTH/NCCD pipeline uses.
3. Resize to the configuration's resolution, normalise to [-1, 1] as `(x - 127) / 127`.

Raw Hounsfield slices are extracted **once** and cached as int16. Windowing is then
applied cheaply per experiment, so comparing two windows does not mean re-reading 64
NIfTI volumes twice.

In [ ]:
_clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))

def window_to_uint8(sl_hu, level, width):
    lo, hi = level - width / 2.0, level + width / 2.0
    return (np.clip((sl_hu - lo) / (hi - lo), 0, 1) * 255).astype(np.uint8)

def prep_slice(sl_hu, level, width, size=RAW_SIZE):
    u8 = _clahe.apply(window_to_uint8(sl_hu, level, width))
    if u8.shape != (size, size):
        u8 = cv2.resize(u8, (size, size), interpolation=cv2.INTER_AREA)
    return u8

def to_model_input(u8):
    return ((u8.astype(np.float32) - 127.0) / 127.0)[..., None]

def load_volume(case):
    v = np.asanyarray(nib.load(os.path.join(IMG_DIR, case)).dataobj, dtype=np.float32)
    m = (np.asanyarray(nib.load(os.path.join(LBL_DIR, case)).dataobj) > 0).astype(np.uint8)
    return v, m

In [ ]:
v, m = load_volume(train_c[0])
print("case      :", train_c[0])
print("volume    :", v.shape)
print("HU range  :", float(v.min()), "..", float(v.max()))
print("tumour vox:", int(m.sum()), f"({m.mean()*100:.4f}% of volume)")

# How much tumour actually survives each candidate window without saturating?
tum = v[m > 0]
for w in [LUNG, SOFT]:
    lo, hi = w["level"] - w["width"] / 2, w["level"] + w["width"] / 2
    inside = float(((tum > lo) & (tum < hi)).mean())
    print(f"  {w['wdesc']:20s} [{lo:7.0f},{hi:7.0f}] HU -> "
          f"{inside:6.1%} of tumour voxels fall inside (not clipped)")

### 3.1 Diagnostic: does the legacy lung-ROI hack destroy tumour pixels?

The original IQ-OTH/NCCD preprocessing applies a crude lung ROI before segmentation:
inverse-threshold at 127, erode 4x4, dilate 13x13. With ground-truth tumour masks
available, the fraction of tumour pixels that step discards can be measured directly
for the first time.

In [ ]:
def legacy_lung_roi(u8):
    _, roi = cv2.threshold(u8, 127, 255, cv2.THRESH_BINARY_INV)
    roi = cv2.erode(roi, np.ones([4, 4], np.uint8))
    roi = cv2.dilate(roi, np.ones([13, 13], np.uint8))
    return roi > 0

kept, total, checked = 0, 0, 0
for case in tqdm(train_c[:8], desc="ROI check"):
    v, m = load_volume(case)
    for z in np.where(m.sum(axis=(0, 1)) > 0)[0]:
        u8 = prep_slice(v[:, :, z], -600, 1500, 256)     # the legacy window
        gt = cv2.resize(m[:, :, z], (256, 256), interpolation=cv2.INTER_NEAREST) > 0
        if gt.sum() == 0:
            continue
        kept += int((gt & legacy_lung_roi(u8)).sum()); total += int(gt.sum()); checked += 1

frac = kept / max(total, 1)
print(f"\ntumour-bearing slices checked  : {checked}")
print(f"tumour pixels surviving the ROI: {frac:.1%}")
print("VERDICT:", "destructive, drop it" if frac < 0.80 else
      "broadly safe, though still an untuned heuristic")
RESULTS["legacy_roi_diagnostic"] = {"slices_checked": checked,
                                    "tumour_pixel_retention": float(frac)}
save_json()

## 4. Build the slice dataset

Every tumour-bearing slice is kept, plus `NEG_RATIO` tumour-free slices per positive.

**`NEG_RATIO` is 0.3, not 1.0.** At 1.0 a model that predicts nothing at all still
scores well on any metric that rewards correctly-empty slices, which is precisely how
the first version of this notebook failed. Tumour voxels are under 0.2% of the data, so
the balance has to be managed deliberately and reported.

In [ ]:
def build_slices_raw(case_list, desc):
    X, Y = [], []
    for case in tqdm(case_list, desc=desc):
        v, m = load_volume(case)
        pos = np.where(m.sum(axis=(0, 1)) > 0)[0]
        neg_pool = np.setdiff1d(np.arange(m.shape[2]), pos)
        n_neg = min(len(neg_pool), int(round(NEG_RATIO * len(pos))))
        neg = rng.choice(neg_pool, n_neg, replace=False) if n_neg else np.array([], int)
        for z in np.concatenate([pos, neg]).astype(int):
            sl = cv2.resize(v[:, :, z], (RAW_SIZE, RAW_SIZE), interpolation=cv2.INTER_AREA)
            X.append(np.clip(sl, -1024, 3071).astype(np.int16))     # raw HU, cached once
            Y.append(cv2.resize(m[:, :, z], (RAW_SIZE, RAW_SIZE),
                                interpolation=cv2.INTER_NEAREST))
        del v, m
    return np.stack(X), np.stack(Y)

XtrRAW, Ytr = build_slices_raw(train_c, "train slices")
XvaRAW, Yva = build_slices_raw(val_c, "val slices")
print("train:", XtrRAW.shape, "positive slices:", int((Ytr.sum((1, 2)) > 0).sum()))
print("val  :", XvaRAW.shape, "positive slices:", int((Yva.sum((1, 2)) > 0).sum()))
print("tumour pixel fraction (train): %.5f" % Ytr.mean())

RESULTS["dataset"] = {
    "train_slices": int(len(XtrRAW)), "val_slices": int(len(XvaRAW)),
    "train_positive_slices": int((Ytr.sum((1, 2)) > 0).sum()),
    "tumour_pixel_fraction_train": float(Ytr.mean())}
save_json()

def windowed(XRAW, level, width, size):
    out = np.empty((len(XRAW), size, size), np.uint8)
    for i in range(len(XRAW)):
        u8 = _clahe.apply(window_to_uint8(XRAW[i].astype(np.float32), level, width))
        out[i] = u8 if size == RAW_SIZE else cv2.resize(
            u8, (size, size), interpolation=cv2.INTER_AREA)
    return out

def masks_at(YRAW, size):
    if size == RAW_SIZE:
        return YRAW
    return np.stack([cv2.resize(y, (size, size), interpolation=cv2.INTER_NEAREST)
                     for y in YRAW])

In [ ]:
def augment(x, y):
    if tf.random.uniform([]) < 0.5:
        x, y = tf.image.flip_left_right(x), tf.image.flip_left_right(y)
    if tf.random.uniform([]) < 0.5:
        x, y = tf.image.flip_up_down(x), tf.image.flip_up_down(y)
    k = tf.random.uniform([], 0, 4, dtype=tf.int32)
    return tf.image.rot90(x, k), tf.image.rot90(y, k)

def make_ds(X, Y, training, batch):
    ds = tf.data.Dataset.from_tensor_slices((X, Y))
    ds = ds.map(lambda x, y: (((tf.cast(x, tf.float32) - 127.0) / 127.0)[..., None],
                              tf.cast(y, tf.float32)[..., None]),
                num_parallel_calls=tf.data.AUTOTUNE)
    if training:
        ds = ds.shuffle(2048, seed=SEED).map(augment, num_parallel_calls=tf.data.AUTOTUNE)
    return ds.batch(batch).prefetch(tf.data.AUTOTUNE)

## 5. Model, loss, and metric

A standard 4-level U-Net (Ronneberger et al., 2015), built here rather than inherited,
so the architecture is fully specified in the paper.

### What went wrong in the first version, and what changed

The first run of this notebook produced **zero predicted voxels on every test volume**
while reporting a validation Dice of 0.52. Both facts were true, and the gap between
them was entirely an artifact of the metric:

- The metric was a per-sample soft Dice with `smooth=1`. On a tumour-free slice with an
  empty prediction it evaluates to `(0+1)/(0+0+1) = 1.0`. With half the batch
  tumour-free, **an all-background model scores ~0.5 for free.**
- The Dice term in the loss had the same defect, so it supplied almost no gradient
  toward predicting anything, while the BCE term, facing 0.19% positive pixels, was
  minimised most cheaply by predicting zero everywhere.

Three changes:

1. **`dice_pos`** averages Dice over tumour-bearing samples only, with no smoothing
   term in the numerator. An empty prediction scores 0. It cannot be gamed.
2. **Focal-Tversky loss** with `beta > alpha`, penalising false negatives far more
   heavily than false positives, which is the correct asymmetry when the target
   occupies a fraction of a percent of the image.
3. A **hard guard** after training that fails the run if the model predicts nothing,
   so this failure can never again be mistaken for a result.

In [ ]:
def conv_block(x, f):
    for _ in range(2):
        x = layers.Conv2D(f, 3, padding="same", use_bias=False)(x)
        x = layers.BatchNormalization()(x)
        x = layers.Activation("relu")(x)
    return x

def build_unet(size, base=BASE_FILTERS):
    inp = layers.Input((size, size, 1))
    skips, x = [], inp
    for i in range(4):
        x = conv_block(x, base * 2 ** i)
        skips.append(x)
        x = layers.MaxPooling2D(2)(x)
    x = conv_block(x, base * 16)
    for i in reversed(range(4)):
        x = layers.Conv2DTranspose(base * 2 ** i, 2, strides=2, padding="same")(x)
        x = layers.Concatenate()([x, skips[i]])
        x = conv_block(x, base * 2 ** i)
    # float32 output is required under mixed precision for a numerically safe loss
    out = layers.Conv2D(1, 1, activation="sigmoid", dtype="float32")(x)
    return keras.Model(inp, out, name="unet")


def dice_pos(y_true, y_pred):
    # Dice averaged over samples that CONTAIN tumour. No numerator smoothing, so an
    # empty prediction scores 0 rather than 1. This is the metric v1 got wrong.
    yt = tf.cast(y_true, tf.float32); yp = tf.cast(y_pred, tf.float32)
    inter = tf.reduce_sum(yt * yp, axis=[1, 2, 3])
    denom = tf.reduce_sum(yt, [1, 2, 3]) + tf.reduce_sum(yp, [1, 2, 3])
    has = tf.cast(tf.reduce_sum(yt, [1, 2, 3]) > 0, tf.float32)
    d = 2.0 * inter / (denom + 1e-7)
    return tf.reduce_sum(d * has) / (tf.reduce_sum(has) + 1e-7)

def pred_positive_rate(y_true, y_pred):
    # Fraction of pixels predicted positive. If this sits at 0, the model has collapsed.
    return tf.reduce_mean(tf.cast(tf.cast(y_pred, tf.float32) > 0.5, tf.float32))

ALPHA, BETA, GAMMA = 0.3, 0.7, 0.75      # beta > alpha penalises false negatives

def focal_tversky_loss(y_true, y_pred):
    yt = tf.cast(y_true, tf.float32); yp = tf.cast(y_pred, tf.float32)
    tp = tf.reduce_sum(yt * yp)
    fp = tf.reduce_sum((1 - yt) * yp)
    fn = tf.reduce_sum(yt * (1 - yp))
    tv = (tp + 1e-6) / (tp + ALPHA * fp + BETA * fn + 1e-6)
    return tf.pow(1.0 - tv, GAMMA)

def combined_loss(y_true, y_pred):
    bce = tf.reduce_mean(keras.losses.binary_crossentropy(y_true, y_pred))
    return 0.5 * bce + focal_tversky_loss(y_true, y_pred)

print("params:", f"{build_unet(256).count_params():,}")

## 6. Train one model per CT window

Both windows are trained and compared. Which one preserves tumour contrast is an
empirical question, and reporting both is more informative than reporting only the
winner.

In [ ]:
def train_for(cfg):
    print(f"\n{'='*70}\n{cfg['tag']}  {cfg['wdesc']} at {cfg['size']}px\n{'='*70}")
    Xtr = windowed(XtrRAW, cfg["level"], cfg["width"], cfg["size"])
    Xva = windowed(XvaRAW, cfg["level"], cfg["width"], cfg["size"])
    ytr_, yva_ = masks_at(Ytr, cfg["size"]), masks_at(Yva, cfg["size"])
    ds_tr = make_ds(Xtr, ytr_, True, cfg["batch"])
    ds_va = make_ds(Xva, yva_, False, cfg["batch"])

    keras.backend.clear_session(); tf.random.set_seed(SEED)
    model = build_unet(cfg["size"])
    model.compile(optimizer=keras.optimizers.Adam(1e-3), loss=combined_loss,
                  metrics=[dice_pos, pred_positive_rate])
    ckpt = f"{RESULTS_DIR}/unet_msd_{cfg['tag']}.weights.h5"
    cbs = [
        keras.callbacks.ModelCheckpoint(ckpt, monitor="val_dice_pos", mode="max",
                                        save_best_only=True, save_weights_only=True),
        keras.callbacks.ReduceLROnPlateau(monitor="val_dice_pos", mode="max", factor=0.5,
                                          patience=4, min_lr=1e-6),
        keras.callbacks.EarlyStopping(monitor="val_dice_pos", mode="max", patience=10,
                                      restore_best_weights=True),
    ]
    t0 = time.time()
    h = model.fit(ds_tr, validation_data=ds_va, epochs=EPOCHS, callbacks=cbs, verbose=1)
    meta = {"epochs_run": len(h.history["loss"]),
            "best_val_dice_pos": float(max(h.history["val_dice_pos"])),
            "final_val_pred_positive_rate": float(h.history["val_pred_positive_rate"][-1]),
            "minutes": (time.time() - t0) / 60, "ckpt": ckpt, "history": h.history}
    print(f"\nbest val dice_pos      : {meta['best_val_dice_pos']:.4f}")
    print(f"val pred-positive rate : {meta['final_val_pred_positive_rate']:.6f}")
    if meta["final_val_pred_positive_rate"] < 1e-6:
        print("  COLLAPSED: the model predicts no positive pixels at all.")
    del Xtr, Xva, ytr_, yva_
    return model, meta

runs = {}
for cfg in CONFIGS:
    model, meta = train_for(cfg)
    runs[cfg["tag"]] = {"cfg": cfg,
                        "meta": {k: v for k, v in meta.items() if k != "history"},
                        "history": meta["history"]}
    del model

RESULTS["training"] = {k: v["meta"] for k, v in runs.items()}
save_json()

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for tag, r in runs.items():
    ax[0].plot(r["history"]["loss"], label=f"{tag} train")
    ax[0].plot(r["history"]["val_loss"], "--", label=f"{tag} val")
    ax[1].plot(r["history"]["val_dice_pos"], label=f"{tag} val dice_pos")
ax[0].set_title("combined loss"); ax[0].set_xlabel("epoch"); ax[0].legend(fontsize=8)
ax[1].set_title("validation Dice (tumour-bearing slices only)")
ax[1].set_xlabel("epoch"); ax[1].legend(fontsize=8); ax[1].set_ylim(0, 1)
plt.tight_layout(); plt.savefig(f"{RESULTS_DIR}/fig_training_curves.png", dpi=150); plt.show()

## 7. Evaluation on held-out patients

- **Per-volume 3D Dice, not per-slice.** Averaging slice-level Dice inflates the score,
  because tumour-free slices score well for free.
- **Scored at native resolution.** Predictions are upsampled back to each volume's own
  in-plane size, so training resolution is a modelling choice, not a way of easing the metric.
- **The threshold is selected on validation** and applied unchanged to test.

The sweep now extends down to 0.05, and the maximum predicted probability is reported.
The first version swept only 0.3 upward, which cannot distinguish "the model is
slightly under-confident" from "the model outputs nothing".

In [ ]:
def predict_volume_probs(case, model_, win, size):
    v, m = load_volume(case)
    H, W, D = m.shape
    probs = np.zeros((H, W, D), np.float32)
    step = 32 if size <= 256 else 8
    for s in range(0, D, step):
        zs = list(range(s, min(s + step, D)))
        batch = np.stack([to_model_input(prep_slice(v[:, :, z], win["level"], win["width"], size))
                          for z in zs])
        p = model_.predict(batch, verbose=0)[..., 0].astype(np.float32)
        for k, z in enumerate(zs):
            probs[:, :, z] = cv2.resize(p[k], (W, H), interpolation=cv2.INTER_LINEAR)
    del v
    return probs, m

def volume_scores(pred, gt):
    inter = float((pred & gt).sum()); ps, gs = float(pred.sum()), float(gt.sum())
    union = ps + gs - inter
    return {"dice": (2 * inter / (ps + gs)) if (ps + gs) > 0 else np.nan,
            "iou": (inter / union) if union > 0 else np.nan,
            "gt_voxels": gs, "pred_voxels": ps}

In [ ]:
for tag, r in runs.items():
    keras.backend.clear_session()
    model = build_unet(r["cfg"]["size"]); model.load_weights(r["meta"]["ckpt"])
    acc = {t: [] for t in THRS}
    maxp = 0.0
    for c in tqdm(val_c, desc=f"val [{tag}]"):
        probs, gt = predict_volume_probs(c, model, r["cfg"], r["cfg"]["size"])
        maxp = max(maxp, float(probs.max()))
        for t in THRS:
            acc[t].append(volume_scores((probs >= t).astype(np.uint8), gt)["dice"])
        del probs, gt
    sweep = {t: float(np.nanmean(v)) for t, v in acc.items()}
    best_t = max(sweep, key=sweep.get)
    r["sweep"] = sweep; r["best_thr"] = best_t; r["max_prob"] = maxp
    r["val_dice"] = sweep[best_t]
    print(f"\n[{tag}] max predicted probability anywhere in validation: {maxp:.4f}")
    for t in THRS:
        print(f"   thr {t:.2f} -> val volume Dice {sweep[t]:.4f}")
    print(f"   selected threshold {best_t} (val Dice {sweep[best_t]:.4f})")
    if best_t == max(THRS):
        print("   WARNING: the optimum sits at the TOP of the sweep. The true optimum "
              "is above it and this figure understates the model. Extend THRS.")
    del model

RESULTS["val_sweeps"] = {k: {"sweep": {str(a): b for a, b in v["sweep"].items()},
                             "best_thr": v["best_thr"], "max_prob": v["max_prob"],
                             "val_dice": v["val_dice"]} for k, v in runs.items()}
save_json()

BEST_TAG = max(runs, key=lambda k: runs[k]["val_dice"])
print(f"\nCONFIG SELECTED ON VALIDATION: {BEST_TAG} "
      f"({runs[BEST_TAG]['cfg']['wdesc']} at {runs[BEST_TAG]['cfg']['size']}px)")
print(f"prior v2 soft-tissue window at 256px scored {PRIOR_RESULTS['soft256_val_dice']:.4f} "
      "on validation and is reported in the ablation for completeness")

In [ ]:
assert runs[BEST_TAG]["max_prob"] > 0.05, (
    "The best model never exceeds probability 0.05 anywhere: it has collapsed to "
    "predicting background. Do not report these numbers. Check the training curves, "
    "the tumour-inside-window percentages in section 3, and NEG_RATIO.")
print("guard passed: the model does produce positive predictions")

In [ ]:
best = runs[BEST_TAG]
BEST_SIZE = best["cfg"]["size"]
keras.backend.clear_session()
model = build_unet(BEST_SIZE); model.load_weights(best["meta"]["ckpt"])
THR = best["best_thr"]

rows = []
for c in tqdm(test_c, desc="test volumes"):
    probs, gt = predict_volume_probs(c, model, best["cfg"], BEST_SIZE)
    s = volume_scores((probs >= THR).astype(np.uint8), gt); s["case"] = c
    s["max_prob"] = float(probs.max())
    rows.append(s); del probs, gt

test_df = pd.DataFrame(rows)[["case", "dice", "iou", "gt_voxels", "pred_voxels", "max_prob"]]
print(test_df.round(4).to_string(index=False))

summary = {"config": BEST_TAG, "size": BEST_SIZE, "threshold": THR,
           "n_volumes": int(len(test_df)),
           "dice_mean": float(test_df["dice"].mean()), "dice_std": float(test_df["dice"].std()),
           "dice_median": float(test_df["dice"].median()),
           "iou_mean": float(test_df["iou"].mean()), "iou_std": float(test_df["iou"].std()),
           "iou_median": float(test_df["iou"].median())}
print(f"\n=== HELD-OUT TEST ({BEST_TAG}, thr {THR}) ===")
print(f"Dice  {summary['dice_mean']:.4f} +/- {summary['dice_std']:.4f} "
      f"(median {summary['dice_median']:.4f})")
print(f"IoU   {summary['iou_mean']:.4f} +/- {summary['iou_std']:.4f} "
      f"(median {summary['iou_median']:.4f})")

test_df.to_csv(f"{RESULTS_DIR}/test_per_volume.csv", index=False)
RESULTS["test_summary"] = summary
RESULTS["test_per_volume"] = test_df.to_dict("records")
save_json()
shutil.copy(best["meta"]["ckpt"], f"{RESULTS_DIR}/unet_msd_best.weights.h5")

Report the **distribution**, not only the mean. Lung-tumour Dice is high-variance: a
few small or atypical tumours can pull the mean well below the median, and that spread
is part of the result.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].boxplot([test_df["dice"].dropna(), test_df["iou"].dropna()], labels=["Dice", "IoU"])
ax[0].set_ylim(0, 1); ax[0].grid(alpha=.3)
ax[0].set_title(f"per-volume scores (n={len(test_df)})")
ax[1].scatter(test_df["gt_voxels"], test_df["dice"], alpha=.75)
ax[1].set_xscale("log"); ax[1].set_xlabel("tumour size (GT voxels, log)")
ax[1].set_ylabel("Dice"); ax[1].set_title("Dice vs tumour size"); ax[1].grid(alpha=.3)
plt.tight_layout(); plt.savefig(f"{RESULTS_DIR}/fig_test_distribution.png", dpi=150); plt.show()

## 8. Baseline comparison: the original checkpoint

`UNet_best_Model_checkpoint.h5` scored on the same held-out volumes, forming the first
row of the segmentation ablation. It was trained on unknown data for an unknown
objective in a different domain, so a poor result is expected; the comparison is the
point, and the number is reported whatever it is.

In [ ]:
OLD_URL = ("https://github.com/haseebkhan9081/LViT_Vision_Transformer/"
           "releases/download/weights-v1/UNet_best_Model_checkpoint.h5")
OLD = "/content/UNet_best_Model_checkpoint.h5"
try:
    if not os.path.exists(OLD):
        subprocess.run(["wget", "-q", "-O", OLD, OLD_URL], check=True)
    from tensorflow.keras import backend as K
    def _dc(yt, yp):
        f1, f2 = K.flatten(yt), K.flatten(yp)
        i = K.sum(f1 * f2)
        return (2. * i + 1) / (K.sum(f1) + K.sum(f2) + 1)
    old_model = tf.keras.models.load_model(
        OLD, custom_objects={"dice_coef": _dc, "dice_coef_loss": lambda a, b: -_dc(a, b)},
        compile=False)
    old_size = old_model.input_shape[1] or 512
    old_win = {"level": -600, "width": 1500}     # the window the original pipeline used
    old_rows = []
    for c in tqdm(test_c, desc="old checkpoint"):
        probs, gt = predict_volume_probs(c, old_model, old_win, old_size)
        old_rows.append(dict(volume_scores((probs >= 0.5).astype(np.uint8), gt), case=c))
        del probs, gt
    old_df = pd.DataFrame(old_rows)
    old_sum = {"dice_mean": float(old_df["dice"].mean()),
               "dice_median": float(old_df["dice"].median()),
               "iou_mean": float(old_df["iou"].mean()), "threshold": 0.5}
    print(f"\nold checkpoint: Dice {old_sum['dice_mean']:.4f} "
          f"(median {old_sum['dice_median']:.4f}) | IoU {old_sum['iou_mean']:.4f}")
    RESULTS["old_checkpoint_on_msd"] = old_sum
except Exception as e:
    print("old-checkpoint comparison skipped:", type(e).__name__, e)
    RESULTS["old_checkpoint_on_msd"] = {"error": f"{type(e).__name__}: {e}"}
save_json()

## 9. Qualitative results

In [ ]:
show = test_c[:3]
fig, axes = plt.subplots(len(show), 3, figsize=(10, 3.4 * len(show)))
for row, case in zip(np.atleast_2d(axes), show):
    probs, gt = predict_volume_probs(case, model, best["cfg"], BEST_SIZE)
    pred = (probs >= THR).astype(np.uint8)
    z = int(np.argmax(gt.sum(axis=(0, 1))))
    v, _ = load_volume(case)
    base = prep_slice(v[:, :, z], best["cfg"]["level"], best["cfg"]["width"], gt.shape[0])
    row[0].imshow(base, cmap="bone"); row[0].set_title(f"{case[:14]} z={z}", fontsize=9)
    row[1].imshow(gt[:, :, z], cmap="gray"); row[1].set_title("ground truth", fontsize=9)
    ov = cv2.cvtColor(base, cv2.COLOR_GRAY2RGB)
    for cnts, col in [(cv2.findContours(gt[:, :, z], cv2.RETR_EXTERNAL,
                                        cv2.CHAIN_APPROX_SIMPLE)[0], (0, 255, 0)),
                      (cv2.findContours(pred[:, :, z], cv2.RETR_EXTERNAL,
                                        cv2.CHAIN_APPROX_SIMPLE)[0], (255, 0, 0))]:
        cv2.drawContours(ov, cnts, -1, col, 1)
    row[2].imshow(ov); row[2].set_title("GT (green) vs prediction (red)", fontsize=9)
    for a in row: a.axis("off")
    del probs, gt, v
plt.tight_layout(); plt.savefig(f"{RESULTS_DIR}/fig_qualitative_msd.png", dpi=150); plt.show()

## 10. Transfer check on IQ-OTH/NCCD (qualitative only)

The MSD-trained segmenter applied to the target domain. **No ground truth exists here**,
so no score is produced and none should be invented. The purpose is to make the domain
gap visible in the paper as a figure rather than assumed away.

Optional: requires `KAGGLE_API_TOKEN` (or the legacy `KAGGLE_USERNAME` /
`KAGGLE_KEY` pair) in Colab Secrets, with notebook access enabled. Skipped cleanly if
absent.

In [ ]:
try:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "kagglehub"],
                   check=False)
    from google.colab import userdata
    def _secret(name):
        try:
            v = userdata.get(name)
            return (v or "").strip() or None
        except Exception as e:
            if "NotebookAccess" in type(e).__name__:
                print(f"  {name}: exists but this notebook lacks access "
                      "(Secrets panel -> 'Notebook access' ON)")
            return None
    if _secret("KAGGLE_API_TOKEN"):
        os.environ["KAGGLE_API_TOKEN"] = _secret("KAGGLE_API_TOKEN")
    else:
        os.environ["KAGGLE_USERNAME"] = _secret("KAGGLE_USERNAME")
        os.environ["KAGGLE_KEY"] = _secret("KAGGLE_KEY")
    import kagglehub
    DL = kagglehub.dataset_download("hamdallak/the-iqothnccd-lung-cancer-dataset")
    cand = [d for d, _, _ in os.walk(DL) if os.path.basename(d) == "Malignant cases"]
    IQ = os.path.dirname(cand[0])

    picks = []
    for folder in ["Bengin cases", "Malignant cases", "Normal cases"]:
        picks += [(os.path.join(IQ, folder, f), folder)
                  for f in sorted(os.listdir(os.path.join(IQ, folder)))[:2]]

    fig, axes = plt.subplots(len(picks), 2, figsize=(7, 3.2 * len(picks)))
    for row, (p, folder) in zip(np.atleast_2d(axes), picks):
        g = cv2.cvtColor(cv2.imread(p), cv2.COLOR_BGR2GRAY)
        # already 8-bit, so HU windowing does not apply; CLAHE + resize keeps the rest
        u8 = _clahe.apply(cv2.resize(g, (BEST_SIZE, BEST_SIZE), interpolation=cv2.INTER_AREA))
        pr = model.predict(to_model_input(u8)[None], verbose=0)[0, ..., 0]
        mk = (pr >= THR).astype(np.uint8)
        row[0].imshow(u8, cmap="bone"); row[0].set_title(folder, fontsize=9)
        ov = cv2.cvtColor(u8, cv2.COLOR_GRAY2RGB)
        cnts, _ = cv2.findContours(mk, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        cv2.drawContours(ov, cnts, -1, (255, 0, 0), 1)
        row[1].imshow(ov)
        row[1].set_title(f"prediction ({mk.mean()*100:.2f}% area, max p={pr.max():.2f})",
                         fontsize=9)
        for a in row: a.axis("off")
    plt.tight_layout(); plt.savefig(f"{RESULTS_DIR}/fig_transfer_iqothnccd.png", dpi=150)
    plt.show()
    print("Qualitative only. No ground truth exists here; no score is reported.")
except Exception as e:
    print("transfer check skipped:", type(e).__name__, e)

## 11. Segmentation ablation

In [ ]:
rows = [{"model": "Original UNet_best_Model_checkpoint.h5 (unknown training data)",
         "window": "lung", "dice_mean": RESULTS.get("old_checkpoint_on_msd", {}).get("dice_mean"),
         "iou_mean": RESULTS.get("old_checkpoint_on_msd", {}).get("iou_mean")}]
for tag, r in runs.items():
    rows.append({"model": f"U-Net retrained on MSD Task06 ({r['cfg']['size']}px, focal-Tversky)",
                 "window": r["cfg"]["wdesc"],
                 "dice_mean": summary["dice_mean"] if tag == BEST_TAG else None,
                 "iou_mean": summary["iou_mean"] if tag == BEST_TAG else None,
                 "val_dice": r["val_dice"]})

tbl = pd.DataFrame(rows).round(4)
print(tbl.to_string(index=False))
print("\n(only the validation-selected window is scored on test, by design)")
tbl.to_csv(f"{RESULTS_DIR}/segmentation_ablation.csv", index=False)
with open(f"{RESULTS_DIR}/segmentation_ablation.md", "w") as f:
    f.write(tbl.to_markdown(index=False))
RESULTS["segmentation_ablation"] = tbl.to_dict("records")
save_json()

shutil.make_archive("/content/fyp_phase2s_results", "zip", RESULTS_DIR)
print("\n", sorted(os.listdir(RESULTS_DIR)))
try:
    from google.colab import files
    files.download("/content/fyp_phase2s_results.zip")
except Exception as e:
    print("download manually from the file browser:", e)

## 12. Output and validity checks

`fyp_phase2s_results.zip` contains `results.json`, the selected weights
(`unet_msd_best.weights.h5`), per-volume test scores, the patient split, and all figures.

Conditions that invalidate the run:

- **The collapse guard in section 7 must pass.** If the model never exceeds probability
  0.05 anywhere, it has learned to predict background and no Dice figure from the run
  means anything. This is not a hypothetical: the first version of this notebook failed
  exactly this way while reporting a validation Dice of 0.52.
- **`pred_voxels` in `test_per_volume.csv` must be non-zero.** An all-zero column is the
  same collapse reaching the test set.
- **The threshold must come from validation.** Selecting it against test scores would
  reproduce, in a subtler form, the leakage documented in Phase 0.

Known failure modes:

- **Both windows collapse.** Lower `NEG_RATIO` further (0.1), raise `BETA` toward 0.9 to
  penalise false negatives harder, or train at 512px so small tumours survive
  downsampling.
- **Session terminated mid-training.** Weights are checkpointed on every validation
  improvement, so progress survives; copy them out before rerunning.
- **Download stalls.** The `curl | tar` stream cannot resume. Re-run the cell; it skips
  if `labelsTr` already exists.